# XBeach 2D Case Setup with `xbeachpy`

This notebook walks through a complete 2D XBeach case preparation using the `oceanicospy.models.xbeachpy` subpackage. The example is based on a **non-stationary wave** simulation over a 2D coastal domain (Sound Bay, Caribbean coast of Colombia, May 2025).

## Workflow overview

```
Initializer
    └── create_folders_l1()        # build input / pros / run / output directories
    └── replace_ini_data()         # stamp params.txt with case-level flags

preprocess
    ├── GridMaker.rectangular()    # derive 2D grid from a shapefile bounding box
    ├── BathyMaker.xyz2asc()       # convert TopoBathy CSV  →  XBeach .dep file
    ├── WindForcing                # download ERA5  →  write ASCII wind forcing
    ├── WaterLevelForcing          # download UHSLC gauge data  →  write ASCII tide
    └── BoundaryConditions         # convert SWAN spectra  →  filelist + loclist

execution
    └── CaseRunner                 # write output config, SLURM script, fill params

postprocess
    └── OutputReader               # lazy-open the NetCDF result from XBeach
```

---
**Required input files (place in `<path_case>/input/`):**

| File | Description |
|---|---|
| `XBeach_domain.shp` | Polygon shapefile defining the 2D model extent |
| `TopoBathy*.csv` | X, Y, Z topobathymetry on a regular grid |
| `SpecSWAN.out` | SWAN spectral output at boundary points |
| `points_output.txt` | (optional) list of gauge output X/Y coordinates |

ERA5 winds and UHSLC water levels are **downloaded automatically** if not already cached.

## 0. Imports

In [ ]:
from oceanicospy.models import xbeachpy

import datetime as dt
import warnings
warnings.filterwarnings('ignore')

## 1. Case configuration

All user-facing settings live in plain dictionaries. Nothing is hard-coded inside the package for this step — every flag maps directly to an XBeach `params.txt` keyword.

| Key | Meaning |
|---|---|
| `act_morf` | Morphological updating (0 = off) |
| `act_sedtrans` | Sediment transport (0 = off) |
| `act_wavemodel` | Spectral wave model (1 = surfbeat) |
| `dims` | Dimensionality (2 = 2D) |

In [ ]:
# ── path to the case root ─────────────────────────────────────────────────────
path_case = '/scratchsan/medellin/XBeach_runs/SB_May2025_C1/'

# ── simulation period ─────────────────────────────────────────────────────────
ini_date = dt.datetime(2025, 5, 9, 4)
end_date = dt.datetime(2025, 5, 19, 18)

# ── case-level flags written into params.txt ──────────────────────────────────
ini_case_data = dict(
    case_description='2D sound bay - C1.1',
    act_morf=0,
    act_sedtrans=0,
    act_wavemodel=1,
    dims=2
)

# ── computation / output settings ─────────────────────────────────────────────
comp_data_nonstat = dict(
    ini_comp_date=ini_date.strftime('%Y%m%d.%H%M%S'),
    end_comp_date=end_date.strftime('%Y%m%d.%H%M%S'),
    tint_value=3600,    # output interval [s]
    tintg_value=3600,   # global output interval [s]
    D50_value=0.0003    # median grain size [m]
)

# ── wind domain description (used to define the ERA5 download bounding box) ───
wind_dict = dict(
    lon_ll_wind=278,
    lat_ll_wind=12.3,
    meshes_x_wind=24,
    meshes_y_wind=24,
    dx_wind=0.025,
    dy_wind=0.025
)

## 2. Initialization

`Initializer` does two things:

1. **`create_folders_l1()`** — creates `input/`, `pros/`, `run/`, and `output/` under `path_case`.  
   If `run/` already exists it is **wiped and re-created** to avoid stale files from a previous attempt.

2. **`replace_ini_data()`** — copies the bundled `params_base.txt` template into `run/params.txt` and substitutes the flags from `ini_case_data`.  
   Any key not supplied by the user falls back to the package defaults in `xbeachpy/utils/defaults.py`.

In [ ]:
case = xbeachpy.Initializer(
    root_path=path_case,
    dict_ini_data=ini_case_data,
    ini_date=ini_date,
    end_date=end_date
)
case.create_folders_l1()
case.replace_ini_data()

After this step the folder tree looks like:

```
/scratchsan/medellin/XBeach_runs/SB_May2025_C1/
├── input/           ← place your static input files here
├── pros/
├── run/
│   └── params.txt   ← generated from the bundled template
└── output/
```

## 3. Grid generation

`GridMaker.rectangular()` reads the bounding box of `XBeach_domain.shp` and lays out a uniform 2D grid with spacing `dx = dy = 10 m`.  
The output files `x.grd` and `y.grd` are written to `run/`, and the method returns a metadata dict that is fed into `fill_grid_section()` to update `params.txt`.

> **Tip:** if you already have pre-built `.grd` files, place both inside `input/` and `GridMaker.rectangular()` will detect them, copy them to `run/`, and skip grid construction entirely.

In [ ]:
case_grid = xbeachpy.preprocess.GridMaker(init=case, dx=10, dy=10)
dict_grid = case_grid.rectangular(source_file='XBeach_domain.shp')
case_grid.fill_grid_section(dict_grid)

print('Grid metadata written to params.txt:')
print(dict_grid)

**`dict_grid` structure:**
```python
{
    'xfilepath': 'x.grd',
    'yfilepath': 'y.grd',
    'meshes_x' : <int>,   # number of cells in the x-direction
    'meshes_y' : <int>    # number of cells in the y-direction
}
```

### Variable resolution grids

`dx` is not limited to a scalar. You can also pass:

```python
# Piecewise-constant: 5 m resolution up to x=500 m, then 20 m for the rest
GridMaker(init=case, dx={500: 5, 10000: 20}, dy=10)

# Callable: resolution grows linearly with distance from shore
GridMaker(init=case, dx=lambda x: max(5, 0.01 * x), dy=10)
```

## 4. Bathymetry

`BathyMaker.xyz2asc()` reads a **regular-grid** topobathymetry CSV (named `TopoBathy*.csv`) from `input/`, pivots it into a 2D array ordered in XBeach convention, and saves it as an ASCII `.dep` file in `run/`.

**Expected CSV format:**
```
X,Y,Z
762000,1385000,-12.5
762010,1385000,-12.3
...
```
Positive Z = above datum. NaN cells (gaps in the grid) are written as empty fields in the `.dep` file.

In [ ]:
case_bathy = xbeachpy.preprocess.BathyMaker(init=case, filename='bathymetry')
dict_bathy = case_bathy.xyz2asc()
case_bathy.fill_bathy_section(dict_bathy)

print('Bathymetry metadata written to params.txt:')
print(dict_bathy)

## 5. Wind forcing (ERA5)

`WindForcing` integrates directly with `oceanicospy.retrievals.ERA5Downloader`:

- **`get_winds_from_ERA5()`** — downloads hourly U10/V10 from the CDS API for the bounding box defined in `wind_dict`. If the NetCDF file already exists in `input/` the download is skipped.
- **`write_ERA5_ascii()`** — converts the NetCDF to a two-column (time [s], speed [m/s], direction [°N]) ASCII file and copies/links it into `run/`.  
  The optional `(lon_target, lat_target)` arguments pin the extraction to the nearest ERA5 grid cell. If omitted, the last spatial point in the dataset is used.

> `use_link=False` copies the file; `use_link=True` creates a symlink (saves space on HPC scratch filesystems).

In [ ]:
case_winds = xbeachpy.preprocess.WindForcing(
    init=case,
    wind_info=wind_dict,
    use_link=False
)
case_winds.get_winds_from_ERA5(difference_to_UTC=-5)

dict_winds = case_winds.write_ERA5_ascii(
    era5_filename='winds_era5.nc',
    ascii_filename='winds.wnd',
    lon_target=-81.706,
    lat_target=12.5204
)
case_winds.fill_wind_section(dict_winds)

print('Wind metadata written to params.txt:')
print(dict_winds)

## 6. Water level forcing (UHSLC)

`WaterLevelForcing` connects to the **University of Hawaii Sea Level Center (UHSLC)** research-quality gauge archive:

- Station `737` = Cartagena de Indias tidal gauge, Colombia.
- Data are downloaded in CSV format and converted to a two-column ASCII (time [s], η [m]) file.
- An internal UTC-5 correction is applied automatically.  
  *(See Known limitations for portability considerations.)*

In [ ]:
case_wl = xbeachpy.preprocess.WaterLevelForcing(init=case, use_link=False)
case_wl.get_waterlevel_from_UHSLC(station_code=737)

dict_wl = case_wl.write_UHSLC_ascii(
    UHSLC_filename='h737.csv',
    ascii_filename='water_levels.wl'
)
case_wl.fill_wl_section(dict_wl)

print('Water-level metadata written to params.txt:')
print(dict_wl)

## 7. Wave boundary conditions (SWAN spectra)

`BoundaryConditions.spectra_from_swan()` reads a SWAN spectral output file (`SpecSWAN.out`) and:

1. Splits the multi-site, multi-time file into individual `.sp2` files under `run/bounds_conds/point_<i>/`.
2. Writes a `filelist_<i>.txt` per offshore site (hourly, `dt = 3600 s`, `gamma_break = 0.2`).
3. Assembles a `loclist.txt` mapping each boundary location to its filelist.

The returned dict carries the keys `w_bc_version`, `n_spectrum_loc`, and `bcfilepath`, which go straight into `params.txt`.

> **Note:** the first 3 SWAN sites are treated as nearshore diagnostic points and are **excluded** from the XBeach boundary forcing. See *Known limitations* for context.

In [ ]:
case_bounds = xbeachpy.preprocess.BoundaryConditions(init=case)
bounds_dict = case_bounds.spectra_from_swan(input_filename='SpecSWAN')
case_bounds.fill_boundaries_section(bounds_dict)

print('Boundary condition metadata written to params.txt:')
print(bounds_dict)

## 8. Output configuration and SLURM submission script

`CaseRunner` finalises `params.txt` with output and computation settings:

| Method | Purpose |
|---|---|
| `write_output_file()` | set the output NetCDF filename |
| `write_output_points()` | read gauge coordinates from `points_output.txt` and embed them |
| `select_global_vars()` | choose 2D/3D field variables written at `tintg` |
| `select_point_vars()` | choose time-series variables at the gauge locations |
| `fill_slurm_file()` | copy the SLURM template and stamp paths + case name |
| `fill_computation_section()` | compute `tstop` from the start/end dates and write remaining params |

> `fill_computation_section()` must be called **last** — it converts the datetime strings to a `tstop` value (seconds) and finalises the file.

In [ ]:
case_output = xbeachpy.execution.CaseRunner(
    init=case,
    dict_comp_data=comp_data_nonstat
)

case_output.write_output_file(filename='SoundBay2D.nc')
case_output.write_output_points(filename='points_output.txt')
case_output.fill_slurm_file(case_name='May2025_C1')

case_output.select_global_vars(list_vars=['zs', 'hh', 'zb0', 'H', 'u', 'v'])
case_output.select_point_vars(list_vars=['zs', 'hh', 'zb0', 'H', 'u', 'v'])

case_output.fill_computation_section()   # must be last

print('params.txt is complete. Run folder ready at:', case.dict_folders['run'])

Submit the job from the run folder:
```bash
cd /scratchsan/medellin/XBeach_runs/SB_May2025_C1/run/
sbatch launcher_xbeach.slurm
```

## 9. Post-processing (after the run)

Once XBeach has finished, `OutputReader` lazily opens the NetCDF output and splits it into:
- **field output** — variables with more than 2 dimensions (time × y × x)
- **point output** — variables with exactly 2 dimensions (time × n_points)

In [ ]:
output_path = f"{case.dict_folders['output']}SoundBay2D.nc"

reader = xbeachpy.postprocess.OutputReader(filepath=output_path)

# 2D/3D field variables  (time x y x)
field_ds = reader.read_field_output()

# Point time-series variables  (time x n_points)
point_ds = reader.read_point_output()

In [ ]:
import matplotlib.pyplot as plt

# Significant wave height at the last output time step
H_final = field_ds['H'].isel(globaltime=-1)

fig, ax = plt.subplots(figsize=(9, 6))
H_final.plot(ax=ax, cmap='viridis', robust=True)
ax.set_title('Significant wave height — final time step')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
plt.tight_layout()
plt.show()

In [ ]:
# Water surface elevation time series at gauge 0
zs_p0 = point_ds['zs'].isel(points=0)

fig, ax = plt.subplots(figsize=(11, 3))
zs_p0.plot(ax=ax, color='steelblue')
ax.set_title('Water surface elevation — gauge 0')
ax.set_xlabel('time')
ax.set_ylabel('zs [m]')
plt.tight_layout()
plt.show()

---
## 10. Package strengths and known limitations

### Strengths

| # | Strength | Where |
|---|---|---|
| 1 | **Automated data retrieval with caching** — ERA5 winds and UHSLC water levels are downloaded on demand; a re-run detects the cached file and skips the API call. | `WindForcing`, `WaterLevelForcing` |
| 2 | **Single `init` object as the backbone** — folder paths, date range, and physics flags are defined once in `Initializer` and shared by every preprocessing class, eliminating repetition and keeping configs consistent. | `Initializer` |
| 3 | **Flexible grid resolution** — `dx` accepts a scalar, a list of spacings, a piecewise-constant dict `{x_to: dx}`, or any callable `f(x) → dx`. The same interface covers 1D and 2D grids. | `GridMaker._build_variable_dx_axis()` |
| 4 | **Pre-existing grid passthrough** — if `.grd` files are found in `input/` they are standardised and copied to `run/` without rebuilding the grid. | `GridMaker.rectangular()` |
| 5 | **HPC-ready SLURM integration** — a ready-to-submit launcher script is stamped with output paths and case name in one call, with no manual editing required. | `CaseRunner.fill_slurm_file()` |
| 6 | **Symlink or copy mode** — `use_link=True/False` lets you trade disk space for portability on shared HPC scratch filesystems. | `WindForcing`, `WaterLevelForcing` |
| 7 | **KDTree-based 1D bathymetry extraction** — corridor pre-filtering + nearest-neighbour KDTree sampling efficiently handles sparse or noisy XYZ point clouds for 1D profile cases. | `BathyMaker.build_1d_profile_from_grid()` |

---

### Known limitations

| # | Limitation | Location | Suggested fix |
|---|---|---|---|
| 1 | **Hardcoded UTC-5 offset** in the UHSLC converter makes it non-portable to stations in other time zones without editing the source. | `WaterLevelForcing._UHSLC_csv_to_ascii` | Expose `difference_to_UTC` as a constructor or method argument (same pattern already used in `WindForcing`). |
| 2 | **First 3 SWAN sites always skipped** (`if idx_site >= 3`) in the spectral boundary converter. This is correct for the Sound Bay setup but will silently drop spectra if a different SWAN output has fewer offshore sites. | `BoundaryConditions.spectra_from_swan` | Add an `n_skip` parameter (default 3) or let the user supply `offshore_points` explicitly. |
| 3 | **`xyz2asc()` assumes a regular grid** — the `pivot_table` step produces NaN-filled rows/columns for scattered or irregular XYZ clouds, leading to a malformed `.dep` file. | `BathyMaker.xyz2asc` | Add a scattered-to-regular interpolation step (e.g. `scipy.interpolate.griddata`) as a fallback for non-regular inputs. |
| 4 | **Wind direction formula flagged as unverified** in the source code (`# this needs to be verified somehow`). The nautical convention `(270 − θ_cart) % 360` may need adjustment depending on the ERA5 u/v axis orientation at a given site. | `WindForcing._ERA5_nc_to_ascii` | Validate against a known NOAA/ECMWF record and add a unit test. |
| 5 | **`OutputReader` is minimal** — variables are separated only by array rank (ndim > 2 vs. ndim == 2). There are no time-slicing, spatial-subsetting, or variable-lookup helpers. | `postprocess.OutputReader` | Add convenience methods such as `get_var(name)`, `sel_time(t0, t1)`, and thin plotting wrappers. |
| 6 | **No CRS management** — the grid, bathymetry, and wind domain are assumed to share the same coordinate reference system. A geographic/projected mismatch produces silently incorrect output. | `GridMaker`, `BathyMaker` | Accept an optional `crs` argument and use `pyproj` to reproject inputs before grid assembly. |